# Module 3 — Create Anomaly-Enhanced Dataset

This notebook creates a **Module 3-only** copy of the POS dataset with more labeled anomalies.

It does **not** modify the original dataset used by Module 1, Module 2, or Module 4.

Output folder:

```text
../datasets/module3_anomaly/
```

## 1. Imports and configuration

In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

SEED = 2026
rng = np.random.default_rng(SEED)

# -----------------------------------------------------------------------------
# Directory configuration
# This notebook reads the original shared datasets and creates a Module 3-only copy.


# Notebook location:
# Notebooks/Module 3/Data Generation/00_create_module3_anomaly_dataset.ipynb

# Original datasets are here:
# Notebooks/datasets/
INPUT_DIR = Path('../../datasets')

# Module 3-only enriched dataset will be saved here:
# Notebooks/Module 3/datasets/
OUTPUT_DIR = Path('../datasets')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POS_PATH = INPUT_DIR / 'enterprise_pos_dataset.csv'
ANOMALIES_PATH = INPUT_DIR / 'anomalies_ground_truth.csv'
CASHIERS_PATH = INPUT_DIR / 'cashiers.csv'
CUSTOMERS_PATH = INPUT_DIR / 'customers.csv'
DAILY_DEMAND_PATH = INPUT_DIR / 'daily_item_demand.csv'

OUT_POS_PATH = OUTPUT_DIR / 'enterprise_pos_dataset.csv'
OUT_ANOMALIES_PATH = OUTPUT_DIR / 'anomalies_ground_truth.csv'
OUT_CASHIERS_PATH = OUTPUT_DIR / 'cashiers.csv'
OUT_CUSTOMERS_PATH = OUTPUT_DIR / 'customers.csv'
OUT_DAILY_DEMAND_PATH = OUTPUT_DIR / 'daily_item_demand.csv'
OUT_REPORT_PATH = OUTPUT_DIR / 'anomaly_injection_report.csv'
OUT_SUMMARY_PATH = OUTPUT_DIR / 'module3_anomaly_summary.csv'

print('Input dir:', INPUT_DIR.resolve())
print('Output dir:', OUTPUT_DIR.resolve())

# Target counts for the Module 3-only anomaly-enriched dataset.
# The goal is to keep anomalies rare, but increase them enough for stable evaluation.
TARGET_COUNTS = {
    'suspicious_discount': 240,
    'void_after_payment': 200,
    'basket_size_outlier': 200,
    'price_tampering': 180,
    'odd_hour': 170,
    'shift_end_void_cluster': 120,
}


Input dir: C:\Users\slimc\Desktop\Notebooks\datasets
Output dir: C:\Users\slimc\Desktop\Notebooks\Module 3\datasets


## 2. Load original datasets

In [4]:

def check_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    return path

for path in [POS_PATH, ANOMALIES_PATH, CASHIERS_PATH, CUSTOMERS_PATH, DAILY_DEMAND_PATH]:
    check_file(path)

pos = pd.read_csv(POS_PATH, sep='|')
anomalies = pd.read_csv(ANOMALIES_PATH)
cashiers = pd.read_csv(CASHIERS_PATH)
customers = pd.read_csv(CUSTOMERS_PATH)
daily_demand = pd.read_csv(DAILY_DEMAND_PATH)

print('Original POS shape:', pos.shape)
print('Original unique orders:', pos['order_id'].nunique())
print('Original anomalies:', len(anomalies))
print('Original anomaly rate:', f"{len(anomalies) / pos['order_id'].nunique():.3%}")
display(anomalies['anomaly_type'].value_counts().to_frame('original_count'))


Original POS shape: (178839, 17)
Original unique orders: 63049
Original anomalies: 468
Original anomaly rate: 0.742%


,original_count
suspicious_discount,125
void_after_payment,94
basket_size_outlier,94
price_tampering,62
odd_hour,62
shift_end_void_cluster,31


## 3. Helper functions

In [5]:

def choose_orders(candidates, n):
    candidates = list(candidates)
    if n <= 0:
        return []
    if len(candidates) < n:
        print(f'Warning: requested {n} orders but only {len(candidates)} candidates exist. Using all candidates.')
        n = len(candidates)
    return rng.choice(candidates, size=n, replace=False).astype(int).tolist()


def build_order_indices(df):
    return {int(k): np.asarray(v, dtype=int) for k, v in df.groupby('order_id', sort=False).indices.items()}


def recompute_daily_demand(pos_mod, daily_original):
    """Recompute quantity and average price after Module 3-only anomaly injection."""
    daily_mod = daily_original.copy()
    sold = (
        pos_mod.groupby(['order_date', 'item_id'], as_index=False)
        .agg(quantity_new=('order_details_id', 'count'), avg_price_new=('price', 'mean'))
    )
    daily_mod = daily_mod.merge(sold, on=['order_date', 'item_id'], how='left')
    daily_mod['quantity'] = daily_mod['quantity_new'].fillna(0).astype(int)
    daily_mod['avg_price'] = daily_mod['avg_price_new'].fillna(daily_mod['avg_price'])
    daily_mod = daily_mod.drop(columns=['quantity_new', 'avg_price_new'])
    return daily_mod


## 4. Select additional anomalies to inject

The new labels are selected only from orders that are currently normal, so existing anomaly labels are preserved.

In [6]:

pos_mod = pos.copy()
anomalies_mod = anomalies.copy()

order_indices = build_order_indices(pos_mod)
existing_anomalous = set(anomalies_mod['order_id'].astype(int))
all_order_ids = list(order_indices.keys())
available_set = {oid for oid in all_order_ids if oid not in existing_anomalous}

current_counts = anomalies_mod['anomaly_type'].value_counts().to_dict()
additional_needed = {typ: max(0, target - current_counts.get(typ, 0)) for typ, target in TARGET_COUNTS.items()}

print('Additional anomalies to inject for Module 3 only:')
display(pd.Series(additional_needed, name='additional_needed').to_frame())

new_logs = []
selected_by_type = {}


def take_candidates(anomaly_type, candidates=None):
    global available_set
    pool = list(available_set if candidates is None else (set(candidates) & available_set))
    selected = choose_orders(pool, additional_needed[anomaly_type])
    selected_set = set(selected)
    available_set -= selected_set
    selected_by_type[anomaly_type] = selected
    for oid in selected:
        new_logs.append({
            'order_id': int(oid),
            'anomaly_type': anomaly_type,
            'description': f'Injected module3-only {anomaly_type} anomaly'
        })
    print(f'{anomaly_type}: injected {len(selected)} new labels')
    return selected


Additional anomalies to inject for Module 3 only:


,additional_needed
suspicious_discount,115
void_after_payment,106
basket_size_outlier,106
price_tampering,118
odd_hour,108
shift_end_void_cluster,89


## 5. Inject Module 3-only anomalies

This modifies only the in-memory copy `pos_mod`, not the original dataset.

In [7]:

# 1) suspicious_discount
selected = take_candidates('suspicious_discount')
discounts = rng.choice([70.0, 75.0, 80.0, 85.0, 90.0, 95.0], size=len(selected), replace=True)
for oid, pct in zip(selected, discounts):
    idxs = order_indices[oid]
    pos_mod.loc[idxs, 'discount_pct'] = pct
    pos_mod.loc[idxs, 'line_total'] = (pos_mod.loc[idxs, 'price'] * (1 - pct / 100)).round(2)
    if rng.random() < 0.45:
        pos_mod.loc[idxs, 'cashier_id'] = 'C07'

# 2) void_after_payment
selected = take_candidates('void_after_payment')
for oid in selected:
    idxs = order_indices[oid]
    pos_mod.loc[idxs, 'is_voided'] = True
    pos_mod.loc[idxs, 'void_reason'] = rng.choice(['customer_complaint', 'manual_correction', 'unknown'])
    if rng.random() < 0.55:
        pos_mod.loc[idxs, 'cashier_id'] = 'C07'

# 3) basket_size_outlier by adding extra lines to existing normal orders
selected = take_candidates('basket_size_outlier')
next_detail_id = int(pos_mod['order_details_id'].max()) + 1
menu_lookup = {
    section: g[['restaurant_type', 'item_name', 'category', 'item_id', 'price']].drop_duplicates().to_dict('records')
    for section, g in pos_mod.groupby('restaurant_type')
}
new_rows = []

for oid in selected:
    idxs = order_indices[oid]
    order_rows = pos_mod.loc[idxs]
    template = order_rows.iloc[0].to_dict()
    section_items = menu_lookup.get(template['restaurant_type'], [])
    if not section_items:
        continue

    current_size = len(order_rows)
    target_size = int(rng.integers(15, 23))
    n_extra = max(0, target_size - current_size)

    for _ in range(n_extra):
        item = section_items[int(rng.integers(0, len(section_items)))]
        row = template.copy()
        row['order_details_id'] = next_detail_id
        row['item_name'] = item['item_name']
        row['category'] = item['category']
        row['item_id'] = item['item_id']
        row['price'] = float(item['price'])
        row['discount_pct'] = 0.0
        row['line_total'] = round(float(item['price']), 2)
        row['is_voided'] = False
        row['void_reason'] = ''
        new_rows.append(row)
        next_detail_id += 1

if new_rows:
    pos_mod = pd.concat([pos_mod, pd.DataFrame(new_rows)], ignore_index=True)

# 4) price_tampering
selected = take_candidates('price_tampering')
for oid in selected:
    idxs = order_indices[oid]
    idx = int(idxs[int(rng.integers(0, len(idxs)))])
    original_price = float(pos_mod.at[idx, 'price'])
    tampered = round(original_price * float(rng.uniform(0.25, 0.55)), 2)
    pos_mod.at[idx, 'price'] = tampered
    discount = float(pos_mod.at[idx, 'discount_pct']) if pd.notna(pos_mod.at[idx, 'discount_pct']) else 0.0
    pos_mod.at[idx, 'line_total'] = round(tampered * (1 - discount / 100), 2)

# 5) odd_hour
selected = take_candidates('odd_hour')
odd_hours = ['03:00 AM', '03:30 AM', '04:00 AM', '04:30 AM', '11:30 PM', '12:30 AM']
for oid in selected:
    idxs = order_indices[oid]
    pos_mod.loc[idxs, 'order_time'] = rng.choice(odd_hours)

# 6) shift_end_void_cluster
late_times = ['01:00 PM', '08:00 PM', '08:30 PM', '09:00 PM']
late_candidates = pos_mod.loc[pos_mod['order_time'].isin(late_times), 'order_id'].drop_duplicates().astype(int).tolist()
selected = take_candidates('shift_end_void_cluster', late_candidates)
for oid in selected:
    idxs = order_indices[oid]
    pos_mod.loc[idxs, 'is_voided'] = True
    pos_mod.loc[idxs, 'void_reason'] = 'item_unavailable'
    pos_mod.loc[idxs, 'cashier_id'] = 'C07'


suspicious_discount: injected 115 new labels
void_after_payment: injected 106 new labels
basket_size_outlier: injected 106 new labels
price_tampering: injected 118 new labels
odd_hour: injected 108 new labels
shift_end_void_cluster: injected 89 new labels


## 6. Save Module 3-only files

In [8]:

# Build final Module 3-only anomaly labels
new_logs_df = pd.DataFrame(new_logs)
anomalies_mod = pd.concat([anomalies_mod, new_logs_df], ignore_index=True)
anomalies_mod = anomalies_mod.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)

# Recompute daily demand so the Module 3 copy stays internally consistent
daily_mod = recompute_daily_demand(pos_mod, daily_demand)

# Save with the same filenames inside ../datasets/module3_anomaly.
# This makes Notebook 1 easy to update: only INPUT_DIR changes.
pos_mod = pos_mod.sort_values(['order_date', 'order_id', 'order_details_id']).reset_index(drop=True)
pos_mod.to_csv(MODULE3_INPUT_DIR / 'enterprise_pos_dataset.csv', sep='|', index=False)
anomalies_mod.to_csv(MODULE3_INPUT_DIR / 'anomalies_ground_truth.csv', index=False)
cashiers.to_csv(MODULE3_INPUT_DIR / 'cashiers.csv', index=False)
customers.to_csv(MODULE3_INPUT_DIR / 'customers.csv', index=False)
daily_mod.to_csv(MODULE3_INPUT_DIR / 'daily_item_demand.csv', index=False)
new_logs_df.to_csv(MODULE3_INPUT_DIR / 'anomaly_injection_report.csv', index=False)

summary = anomalies_mod['anomaly_type'].value_counts().rename_axis('anomaly_type').reset_index(name='count')
summary['anomaly_rate_orders'] = summary['count'] / pos_mod['order_id'].nunique()
summary.to_csv(MODULE3_INPUT_DIR / 'module3_anomaly_summary.csv', index=False)

print('Module 3-only dataset saved to:', MODULE3_INPUT_DIR)
print('Module 3 POS shape:', pos_mod.shape)
print('Module 3 unique orders:', pos_mod['order_id'].nunique())
print('Module 3 anomalies:', len(anomalies_mod))
print('Module 3 anomaly rate:', f"{len(anomalies_mod) / pos_mod['order_id'].nunique():.3%}")
print('\nFinal anomaly counts:')
display(anomalies_mod['anomaly_type'].value_counts().to_frame('count'))
print('\nGenerated files:')
for p in sorted(MODULE3_INPUT_DIR.glob('*')):
    print(' -', p)


Module 3-only dataset saved to: ..\datasets\module3_anomaly
Module 3 POS shape: (180519, 17)
Module 3 unique orders: 63049
Module 3 anomalies: 1110
Module 3 anomaly rate: 1.761%

Final anomaly counts:


,count
suspicious_discount,240
void_after_payment,200
basket_size_outlier,200
price_tampering,180
odd_hour,170
shift_end_void_cluster,120



Generated files:
 - ..\datasets\module3_anomaly\anomalies_ground_truth.csv
 - ..\datasets\module3_anomaly\anomaly_injection_report.csv
 - ..\datasets\module3_anomaly\cashiers.csv
 - ..\datasets\module3_anomaly\customers.csv
 - ..\datasets\module3_anomaly\daily_item_demand.csv
 - ..\datasets\module3_anomaly\enterprise_pos_dataset.csv
 - ..\datasets\module3_anomaly\module3_anomaly_summary.csv


## Next step

Run the updated Notebook 1 using:

```python
INPUT_DIR = '../datasets/module3_anomaly'
```

Then rerun Notebook 2 on the newly prepared order-level dataset.